In [ ]:
!pip install -q umap-learn leidenalg igraph scanpy louvain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 35.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 49.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 50.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 55.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 18.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.3 MB/s eta 0:00:00
   ━━━━━━

In [ ]:
import umap, scanpy, leidenalg
print("umap:", umap.__version__)
print("scanpy:", scanpy.__version__)
print("All good ✓")

umap: 0.5.12
scanpy: 1.12.1
All good ✓


/tmp/ipykernel_9399/490662050.py:3: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", scanpy.__version__)


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving sample_submission.csv to sample_submission.csv
Saving train.csv to train.csv
Saving test.csv to test.csv


In [ ]:
###############################################################################
# ELL409 Project 2 – Clustering Pipeline v5
# ==========================================
# Fixes from v4 diagnostics:
#   - Part B: resolution scan was BROKEN (all resolutions gave k>20, fell back
#     to 35 clusters). Fix: scan LOWER resolutions (0.2-0.8) and widen range.
#   - Part D: silhouette only 0.50, try more gene features + tune spatial weight
#   - Parts A & C: keep identical (they were strong)
###############################################################################

import numpy as np
import pandas as pd
import warnings, os
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("⚠  pip install umap-learn")

try:
    import scanpy as sc
    import anndata as ad
    HAS_SCANPY = True
except ImportError:
    HAS_SCANPY = False
    print("⚠  pip install scanpy leidenalg igraph louvain")

print("✓ Imports done")

###############################################################################
# CONFIG
###############################################################################
DATA_DIR = "."

GENE_COLS    = [f"g{i}" for i in range(1, 501)]
PROT_COLS    = [f"p{i}" for i in range(1, 41)]
MORPH_COLS   = [f"morph{i}" for i in range(1, 31)]
SPATIAL_COLS = ["x_coord", "y_coord", "neighbor_density", "local_heterogeneity"]

###############################################################################
# LOAD DATA
###############################################################################
print("\n── Loading data ──")
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print(f"   train: {train.shape}, test: {test.shape}, sub: {sample_sub.shape}")

###############################################################################
# HELPERS
###############################################################################

def get_features(df, cols):
    present = [c for c in cols if c in df.columns]
    return df[present].fillna(0).values


def variance_filter(X, quantile=0.25):
    v = np.var(X, axis=0)
    thr = np.quantile(v, quantile)
    mask = v > thr
    if mask.sum() < 10:
        mask[:] = True
    return X[:, mask], mask


def pca_reduce(X, n_components=50):
    n = min(n_components, X.shape[1], X.shape[0] - 1)
    return PCA(n_components=n, random_state=42).fit_transform(X)


def umap_reduce(X, n_components=2, n_neighbors=30, min_dist=0.1):
    if not HAS_UMAP:
        return pca_reduce(X, n_components=min(15, X.shape[1]))
    return umap.UMAP(
        n_components=n_components, n_neighbors=n_neighbors,
        min_dist=min_dist, random_state=42, metric='euclidean'
    ).fit_transform(X)


def leiden_cluster(X, n_neighbors=20, resolution=1.0, random_state=42):
    if not HAS_SCANPY:
        from sklearn.mixture import GaussianMixture
        best_bic, best_labels = np.inf, None
        for k in range(5, 21):
            gmm = GaussianMixture(n_components=k, random_state=42, n_init=3)
            gmm.fit(X)
            bic = gmm.bic(X)
            if bic < best_bic:
                best_bic = bic
                best_labels = gmm.predict(X)
        return best_labels
    adata = ad.AnnData(X.astype(np.float32))
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep='X', random_state=random_state)
    sc.tl.leiden(adata, resolution=resolution, random_state=random_state)
    return adata.obs['leiden'].astype(int).values


def relabel(labels):
    uq = np.unique(labels)
    mp = {old: new for new, old in enumerate(uq)}
    return np.array([mp[l] for l in labels])


def scan_resolutions(X_emb, leiden_nn, res_list, target_lo, target_hi):
    """Try multiple resolutions, pick best silhouette within target k range."""
    best_score = -1
    best_labels = None
    best_res = None
    best_k = None
    n_sample = min(3000, X_emb.shape[0])

    print(f"      scanning {len(res_list)} resolutions (target k: {target_lo}-{target_hi}) …")
    for res in res_list:
        labels = leiden_cluster(X_emb, n_neighbors=leiden_nn, resolution=res)
        nc = len(np.unique(labels))

        if nc < target_lo or nc > target_hi or nc < 2:
            print(f"         res={res:.2f} → k={nc} (out of range, skip)")
            continue

        if X_emb.shape[0] > n_sample:
            idx = np.random.RandomState(42).choice(X_emb.shape[0], n_sample, replace=False)
            score = silhouette_score(X_emb[idx], labels[idx])
        else:
            score = silhouette_score(X_emb, labels)

        print(f"         res={res:.2f} → k={nc}, sil={score:.4f}" +
              (" ★" if score > best_score else ""))

        if score > best_score:
            best_score = score
            best_labels = labels
            best_res = res
            best_k = nc

    if best_labels is None:
        print("      ⚠ no resolution in range — using lowest resolution")
        best_labels = leiden_cluster(X_emb, n_neighbors=leiden_nn, resolution=res_list[0])
        best_res = res_list[0]
        best_k = len(np.unique(best_labels))

    print(f"      ✓ selected: res={best_res}, k={best_k}, sil={best_score:.4f}")
    return best_labels


def build_spatial_neighbor_features(df, k=15):
    coords = df[["x_coord", "y_coord"]].values
    nn = NearestNeighbors(n_neighbors=k + 1, metric='euclidean').fit(coords)
    dists, idxs = nn.kneighbors(coords)
    nb_idx = idxs[:, 1:]

    feat_cols = [c for c in PROT_COLS + MORPH_COLS[:10] if c in df.columns]
    F = df[feat_cols].fillna(0).values

    nb_mean = np.zeros((len(df), F.shape[1]))
    nb_std  = np.zeros((len(df), F.shape[1]))
    for i in range(len(df)):
        nb_mean[i] = F[nb_idx[i]].mean(axis=0)
        nb_std[i]  = F[nb_idx[i]].std(axis=0)

    mean_dist = dists[:, 1:].mean(axis=1, keepdims=True)
    return np.hstack([nb_mean, nb_std, mean_dist])


def diagnostics(X_emb, labels, part_name):
    nc = len(np.unique(labels))
    sizes = np.bincount(labels)
    sil = silhouette_score(X_emb[:min(3000, len(X_emb))],
                            labels[:min(3000, len(labels))])
    ch = calinski_harabasz_score(X_emb, labels)
    print(f"\n   📊 Part {part_name} diagnostics:")
    print(f"      clusters: {nc}")
    print(f"      silhouette: {sil:.4f}")
    print(f"      calinski-harabasz: {ch:.1f}")
    print(f"      cluster sizes: {sorted(sizes, reverse=True)}")
    print(f"      smallest: {sizes.min()}, largest: {sizes.max()}")


###############################################################################
# PIPELINES
###############################################################################

def pipeline_A(tr, te):
    """Part A — IDENTICAL to v4 (scored well)."""
    print("\n═══ Part A: Sparse Marker Clustering ═══")
    feat_cols = GENE_COLS + PROT_COLS + MORPH_COLS
    combined = pd.concat([tr, te], ignore_index=True)
    n_tr = len(tr)
    X = get_features(combined, feat_cols)

    X, _ = variance_filter(X, quantile=0.30)
    print(f"   features after var-filter: {X.shape[1]}")
    X = RobustScaler().fit_transform(X)
    X_pca = pca_reduce(X, 50)
    X_emb = umap_reduce(X_pca, n_components=15, n_neighbors=30, min_dist=0.0)

    labels = scan_resolutions(
        X_emb, leiden_nn=20,
        res_list=[0.6, 0.7, 0.8, 0.9, 1.0, 1.1],
        target_lo=5, target_hi=15
    )
    diagnostics(X_emb, labels, "A")
    return relabel(labels[n_tr:])


def pipeline_B(tr, te):
    """Part B — FIXED: scan much lower resolutions since this data fragments easily.
    Also use larger leiden_nn to avoid over-splitting rare clusters.
    """
    print("\n═══ Part B: Rare Cell-Type Discovery (FIXED) ═══")
    feat_cols = GENE_COLS + PROT_COLS + MORPH_COLS
    combined = pd.concat([tr, te], ignore_index=True)
    n_tr = len(tr)
    X = get_features(combined, feat_cols)

    X, _ = variance_filter(X, quantile=0.20)
    print(f"   features after var-filter: {X.shape[1]}")
    X = RobustScaler().fit_transform(X)
    X_pca = pca_reduce(X, 50)

    # Keep small UMAP n_neighbors to preserve rare groups
    X_emb = umap_reduce(X_pca, n_components=15, n_neighbors=15, min_dist=0.0)

    # FIX: scan MUCH lower resolutions (v4 had k=23 even at res=1.0!)
    # Also widen target range since rare cell-type problems can have more clusters
    labels = scan_resolutions(
        X_emb, leiden_nn=15,
        res_list=[0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8],
        target_lo=6, target_hi=20
    )
    diagnostics(X_emb, labels, "B")
    return relabel(labels[n_tr:])


def pipeline_C(tr, te):
    """Part C — IDENTICAL to v4 (scored well)."""
    print("\n═══ Part C: Hierarchical & Continuous States ═══")
    feat_cols = GENE_COLS + PROT_COLS + MORPH_COLS
    combined = pd.concat([tr, te], ignore_index=True)
    n_tr = len(tr)
    X = get_features(combined, feat_cols)

    X, _ = variance_filter(X, quantile=0.20)
    print(f"   features after var-filter: {X.shape[1]}")
    X = RobustScaler().fit_transform(X)
    X_pca = pca_reduce(X, 50)
    X_emb = umap_reduce(X_pca, n_components=20, n_neighbors=25, min_dist=0.0)

    labels = scan_resolutions(
        X_emb, leiden_nn=20,
        res_list=[0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4],
        target_lo=8, target_hi=25
    )
    diagnostics(X_emb, labels, "C")
    return relabel(labels[n_tr:])


def pipeline_D(tr, te):
    """Part D — TUNED: use all 500 genes instead of 100, try different spatial weights."""
    print("\n═══ Part D: Spatial Tissue-Niche Clustering (TUNED) ═══")
    combined = pd.concat([tr, te], ignore_index=True)
    n_tr = len(tr)

    # Use ALL gene features (not just first 100)
    mol_cols = PROT_COLS + MORPH_COLS + GENE_COLS
    X_mol = get_features(combined, mol_cols)
    X_mol, _ = variance_filter(X_mol, quantile=0.25)
    X_mol = RobustScaler().fit_transform(X_mol)
    X_mol_pca = pca_reduce(X_mol, 40)  # more PCA components
    print(f"   mol PCA: {X_mol_pca.shape[1]} components")

    # Spatial features
    X_sp = StandardScaler().fit_transform(combined[SPATIAL_COLS].fillna(0).values)

    # Neighbor features
    print("   building neighbor features …")
    X_nb = StandardScaler().fit_transform(
        build_spatial_neighbor_features(combined, k=15)
    )
    print(f"   neighbor features: {X_nb.shape[1]}")

    # Combine — try giving spatial slightly MORE weight
    alpha = 0.7
    X_all = np.hstack([X_mol_pca, X_sp * alpha, X_nb * alpha])
    X_all = pca_reduce(X_all, 35)

    X_emb = umap_reduce(X_all, n_components=15, n_neighbors=20, min_dist=0.0)

    labels = scan_resolutions(
        X_emb, leiden_nn=20,
        res_list=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4],
        target_lo=8, target_hi=30
    )
    diagnostics(X_emb, labels, "D")
    return relabel(labels[n_tr:])


###############################################################################
# RUN
###############################################################################
print("\n" + "=" * 60)
print("   RUNNING PIPELINES v5")
print("=" * 60)

results = {}

for part, fn in [("A", pipeline_A), ("B", pipeline_B),
                 ("C", pipeline_C), ("D", pipeline_D)]:
    tr_p = train[train["part"] == part].reset_index(drop=True)
    te_p = test[test["part"] == part].reset_index(drop=True)
    print(f"\n   Part {part}: train={len(tr_p)}, test={len(te_p)}")

    labels = fn(tr_p, te_p)
    row_ids = te_p["row_id"].values
    for rid, cid in zip(row_ids, labels):
        results[rid] = int(cid)
    print(f"   ✓ Part {part}: {len(np.unique(labels))} clusters → {len(labels)} rows")

###############################################################################
# SUBMISSION
###############################################################################
print("\n" + "=" * 60)
print("   BUILDING SUBMISSION")
print("=" * 60)

submission = sample_sub.copy()
submission["cluster_id"] = submission["row_id"].map(results)

missing = submission["cluster_id"].isna().sum()
if missing > 0:
    print(f"⚠  {missing} row_ids not matched — filling with 0")
    submission["cluster_id"] = submission["cluster_id"].fillna(0)

submission["cluster_id"] = submission["cluster_id"].astype(int)

out_path = f"{DATA_DIR}/submission.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅  Saved: {out_path}")
for part in ["A", "B", "C", "D"]:
    mask = submission["row_id"].str.contains(f"_{part}_")
    k = submission.loc[mask, "cluster_id"].nunique()
    n = mask.sum()
    print(f"   Part {part}: {n} rows, {k} clusters")

print("\n🎉  Done!")
print("\n📋  PASTE THE DIAGNOSTICS BACK so I can compare with v4!")

✓ Imports done

── Loading data ──
   train: (6600, 581), test: (5800, 581), sub: (5800, 2)

   RUNNING PIPELINES v5

   Part A: train=1400, test=1200

═══ Part A: Sparse Marker Clustering ═══
   features after var-filter: 399
      scanning 6 resolutions (target k: 5-15) …
         res=0.60 → k=9, sil=0.9723 ★
         res=0.70 → k=9, sil=0.9723
         res=0.80 → k=9, sil=0.9723
         res=0.90 → k=9, sil=0.9723
         res=1.00 → k=9, sil=0.9723
         res=1.10 → k=12, sil=0.7016
      ✓ selected: res=0.6, k=9, sil=0.9723

   📊 Part A diagnostics:
      clusters: 9
      silhouette: 0.9723
      calinski-harabasz: 527330.6
      cluster sizes: [np.int64(314), np.int64(300), np.int64(298), np.int64(293), np.int64(288), np.int64(280), np.int64(277), np.int64(275), np.int64(275)]
      smallest: 275, largest: 314
   ✓ Part A: 9 clusters → 1200 rows

   Part B: train=1600, test=1400

═══ Part B: Rare Cell-Type Discovery (FIXED) ═══
   features after var-filter: 456
      scanning 